# Evolving a quantum compiler pass — qBraid AI Gateway

Run an LLM-driven evolutionary search that tries to invent a better **qubit
layout** heuristic than the one it starts with.

This notebook uses the **qBraid AI Gateway** — no GPU, no model to download, and
on a qBraid Lab instance no API key to set up either. For the self-hosted GPU
version see `02_local_gpu_quickstart.ipynb`.

> **This spends real quota/credits.** The run below is capped at $2.

Kernel: **Python 3 [ShinkaEvolve]**.

## 1. Setup

Run from the repository root.

In [ ]:
import os
import pathlib
import sys

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "setup"))
print("working from:", ROOT)

### Credentials — usually nothing to do

The gateway accepts `QBRAID_ACCESS_TOKEN`, which qBraid already exports into
every shell on a Lab instance. Off-platform, set `QBRAID_API_KEY` from
<https://account.qbraid.com/account/api-keys>.

In [ ]:
from check_endpoint import gateway_key

if gateway_key():
    print("qBraid credentials found - nothing to do.")
else:
    import getpass

    os.environ["QBRAID_API_KEY"] = getpass.getpass("qBraid API key: ")
    print("key set")

## 2. Check the endpoint before spending anything

Verifies the gateway is reachable, the model is actually served, a completion
round-trips, and you have quota left. Every failure this catches would otherwise
surface ten minutes into a run as a wall of retries.

In [ ]:
!python setup/check_endpoint.py --gateway

### Check the compatibility shim too

ShinkaEvolve hardcodes `n=1` on every call, and the gateway rejects `n`
outright — so without `qbraid_gateway_compat.py` every request returns HTTP 400
and the run never progresses. `run_evolution.py` installs the shim
automatically; this confirms it works.

In [ ]:
!python qbraid_gateway_compat.py

## 3. What does the seed program score?

The seed is a plausible-looking degree-matching heuristic. It scores **below
1.0**, meaning it is slightly *worse* than not reordering the qubits at all —
that is the headroom the search is aiming at.

This takes ~3 seconds, uses no LLM, and costs nothing.

In [ ]:
!python task/evaluate.py --program_path task/initial.py --results_dir /tmp/seed_eval

## 4. Which models can you reach, and what do they cost?

The gateway publishes its lineup and prices. This is the same call
`qbraid_pricing.py` uses to make the budget cap actually bind.

In [ ]:
from qbraid_pricing import fetch_gateway_pricing

prices = fetch_gateway_pricing(gateway_key())
print(f"{'model':<20} {'$/1M in':>9} {'$/1M out':>10}")
for name, p in sorted(prices.items(), key=lambda kv: kv[1]["input"]):
    print(f"{name:<20} {p['input']:>9.2f} {p['output']:>10.2f}")

## 5. Evolve

30 generations, capped at $2. `gpt-5.4-mini` is a good cost/quality balance;
`gpt-5.4-nano` is cheaper, the `gpt-5.6-*` and `claude-*` models are stronger.

Each generation is an LLM call plus a ~2 second evaluation, so the model is the
bottleneck. Output streams below.

In [ ]:
!python run_evolution.py --endpoint gateway \
    --model gpt-5.4-mini \
    --budget 2.00 \
    --generations 30 \
    --results-dir results/gateway_nb \
    --yes

## 6. What did it find?

Check the **scored fraction** first. A low fraction means the model is failing
the SEARCH/REPLACE diff protocol, not that the task is hard.

In [ ]:
import sqlite3

import pandas as pd

rows = sqlite3.connect("results/gateway_nb/programs.sqlite").execute(
    "SELECT generation, combined_score, correct FROM programs ORDER BY generation"
).fetchall()
df = pd.DataFrame(rows, columns=["generation", "score", "correct"])
print(f"{int(df.correct.sum())}/{len(df)} candidates scored")
print(f"seed {df.score.iloc[0]:.4f}  ->  best {df.score.max():.4f}")
df.head(10)

In [ ]:
import matplotlib.pyplot as plt

ok = df[df.correct == 1]
fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(ok.generation, ok.score, s=28, label="candidate")
ax.plot(ok.generation, ok.score.cummax(), lw=2, color="crimson", label="best so far")
ax.axhline(1.0, ls="--", c="gray", lw=1, label="identity layout")
ax.set_xlabel("generation")
ax.set_ylabel("mean speedup vs identity layout")
ax.set_title("Evolution of the layout heuristic")
ax.legend()
plt.tight_layout()
plt.show()

### The winning program

Only the code between the `EVOLVE-BLOCK` markers was allowed to change.

In [ ]:
best = sqlite3.connect("results/gateway_nb/programs.sqlite").execute(
    "SELECT code, combined_score, text_feedback FROM programs "
    "WHERE correct=1 ORDER BY combined_score DESC LIMIT 1"
).fetchone()
print(f"score: {best[1]:.4f}\n")
print(best[2])

In [ ]:
code_str = best[0]
start = code_str.index("# EVOLVE-BLOCK-START")
end = code_str.index("# EVOLVE-BLOCK-END")
print(code_str[start:end])

## 7. Verify it independently

The evaluator is deterministic, so re-scoring the winner must reproduce exactly
the same number. If it does not, every number in the run is suspect.

In [ ]:
!python task/evaluate.py \
    --program_path results/gateway_nb/best/main.py \
    --results_dir /tmp/verify

## Next

- **Run it longer** — `--generations 100 --budget 10`.
- **Change the problem** — edit `task/benchmarks.py`, then run
  `python task/diagnose_panel.py` to confirm your instances have headroom.
- **Go free** — `02_local_gpu_quickstart.ipynb` serves an open-weight model on a
  qBraid GPU, so inference costs nothing per token.
- **Evolve your own code** — see the `shinka-evolve` skill in `skills/`.